### 연습 문제
- data 폴더 안에 `insurance.csv` 파일 존재
- 종속 변수 `charges`
- torch의 다중 퍼셉트론을 이용하여 회귀 모델 생성 (epoch)
- ML 모델로는 XGBoost를 이용하여 회귀 모델을 생성
- R2 Score를 구해서 어떤 모델이 더 좋은 성능을 가지는가.

In [184]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold

##### 내 풀이

In [98]:
df = pd.read_csv('../data/insurance.csv')
df.head(3)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.9240
1,18,male,33.77,1,no,southeast,1725.5523
2,28,male,33.00,3,no,southeast,4449.4620


In [99]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [100]:
print(df['sex'].unique())
print(df['smoker'].unique())
print(df['region'].unique())

<StringArray>
['female', 'male']
Length: 2, dtype: str
<StringArray>
['yes', 'no']
Length: 2, dtype: str
<StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


In [102]:
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['smoker'] = le.fit_transform(df['smoker'])

In [104]:
df = pd.get_dummies(df, columns = ['region'], drop_first=True)
df.head(3)

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.90,0,1,16884.9240,False,False,True
1,18,1,33.77,1,0,1725.5523,False,True,False
2,28,1,33.00,3,0,4449.4620,False,True,False


In [105]:
df.describe()

,age,sex,bmi,children,smoker,charges
count,1338.000000,1338.000000,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,0.505232,30.663397,1.094918,0.204783,13270.422265
std,14.049960,0.500160,6.098187,1.205493,0.403694,12110.011237
min,18.000000,0.000000,15.960000,0.000000,0.000000,1121.873900
25%,27.000000,0.000000,26.296250,0.000000,0.000000,4740.287150
50%,39.000000,1.000000,30.400000,1.000000,0.000000,9382.033000
75%,51.000000,1.000000,34.693750,2.000000,0.000000,16639.912515
max,64.000000,1.000000,53.130000,5.000000,1.000000,63770.428010


In [106]:
df.corr()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
age,1.000000,-0.020856,0.109272,0.042469,-0.025019,0.299008,-0.000407,-0.011642,0.010016
sex,-0.020856,1.000000,0.046371,0.017163,0.076185,0.057292,-0.011156,0.017117,-0.004184
bmi,0.109272,0.046371,1.000000,0.012759,0.003750,0.198341,-0.135996,0.270025,-0.006205
children,0.042469,0.017163,0.012759,1.000000,0.007673,0.067998,0.024806,-0.023066,0.021914
smoker,-0.025019,0.076185,0.003750,0.007673,1.000000,0.787251,-0.036945,0.068498,-0.036945
charges,0.299008,0.057292,0.198341,0.067998,0.787251,1.000000,-0.039905,0.073982,-0.043210
region_northwest,-0.000407,-0.011156,-0.135996,0.024806,-0.036945,-0.039905,1.000000,-0.346265,-0.320829
region_southeast,-0.011642,0.017117,0.270025,-0.023066,0.068498,0.073982,-0.346265,1.000000,-0.346265
region_southwest,0.010016,-0.004184,-0.006205,0.021914,-0.036945,-0.043210,-0.320829,-0.346265,1.000000


In [ ]:
X = df.drop('charges', axis = 1)
y = df['charges']

In [156]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [157]:
sc = StandardScaler()

X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [158]:
X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

In [159]:
class Reg(nn.Module):
    def __init__(self, _dim):
        super(Reg, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.model(x)

In [160]:
model = Reg(X_test_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

In [161]:
model.train()

for epoch in range(300):
    pred = model(X_train_tensor)
    loss = criterion(pred, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1)%20 == 0:
        print(f'Epoch {epoch+1}, Loss: {round(loss.item(), 6)}')

Epoch 20, Loss: 49968940.0
Epoch 40, Loss: 32337322.0
Epoch 60, Loss: 27052250.0
Epoch 80, Loss: 24854406.0
Epoch 100, Loss: 23697840.0
Epoch 120, Loss: 22829174.0
Epoch 140, Loss: 21964262.0
Epoch 160, Loss: 20801220.0
Epoch 180, Loss: 19583236.0
Epoch 200, Loss: 19203574.0
Epoch 220, Loss: 17356300.0
Epoch 240, Loss: 15880564.0
Epoch 260, Loss: 14222594.0
Epoch 280, Loss: 14936885.0
Epoch 300, Loss: 11191797.0


In [162]:
model.eval()

with torch.no_grad():
    pred = model(X_test_tensor)
    r2 = r2_score(y_test_tensor, pred)
    print(f'R2 Score: {round(r2, 4)}')

R2 Score: 0.8326


In [195]:
pipe = Pipeline(
    [
        ('scaler', StandardScaler()),
        ('model', XGBRegressor(random_state = 42))
    ], verbose=True
)

In [196]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

In [197]:
params = {
    'model__n_estimators' : [100, 200, 300],
    'model__max_depth' : [3, 4, 5],
    'model__learning_rate' : [0.01, 0.05, 0.1],
    'model__subsample' : [0.8, 0.9, 1.0]
}

In [198]:
grid_reg = GridSearchCV(
    estimator= pipe,                # 파이프라인으로 생성한 모델을 지정
    param_grid= params,             # dict 형태로 파라미터의 조합
    scoring = 'r2',           # 베스트 모델 선정 기준
    cv=kfold,
    refit = True,                   # 재학습 여부
    n_jobs = -1,                    # 모든 CPU를 사용
    return_train_score= True,       # 학습 데이터의 성능을 출력
    verbose = 2
)

In [199]:
grid_reg.fit(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
[Pipeline] ............ (step 1 of 2) Processing scaler, total=   0.0s
[Pipeline] ............. (step 2 of 2) Processing model, total=   0.1s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=True)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [3, 4, ...], 'model__n_estimators': [100, 200, ...], 'model__subsample': [0.8, 0.9, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messa

In [203]:
round(grid_reg.score(X_test, y_test), 4)

0.8827

In [202]:
print(grid_reg.best_params_)

{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.9}


##### 강사님 풀이

In [204]:
df = pd.read_csv('../data/insurance.csv')
df.head(3)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.9240
1,18,male,33.77,1,no,southeast,1725.5523
2,28,male,33.00,3,no,southeast,4449.4620


In [205]:
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['smoker'] = le.fit_transform(df['smoker'])
df = pd.get_dummies(df, columns = ['region'], drop_first=True)
df.head(3)

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.90,0,1,16884.9240,False,False,True
1,18,1,33.77,1,0,1725.5523,False,True,False
2,28,1,33.00,3,0,4449.4620,False,True,False


In [206]:
X = df.drop('charges', axis = 1)
y = df['charges']

In [207]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [209]:
sc = StandardScaler()

X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)

In [211]:
# 1차 행렬의 데이터를 2차 행렬로 변환
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

print(y_train_tensor.shape)

torch.Size([1070, 1])


In [212]:
# Tensor에서 제공하는 데이터의 구조를 바꾸는 방식

y_train_tensor2 = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(-1)
y_test_tensor2 = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(-1)

In [213]:
torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

tensor([[ 9193.8389],
        [ 8534.6719],
        [27117.9941],
        ...,
        [11931.1250],
        [46113.5117],
        [10214.6357]])

In [214]:
class Reg_Model(nn.Module):
    def __init__(self, _dim):
        super(Reg_Model, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.model(x)

In [215]:
model = Reg_Model(X_train_tensor.shape[1])

In [216]:
criterion = nn.MSELoss()
optimizer= optim.Adam(model.parameters(), lr=0.01)

In [217]:
epochs = 300

for epoch in range(epochs):
    pred = model(X_train_tensor)            # 모델의 예측값
    loss = criterion(pred, y_train_tensor)  # 예측값과 실제값의 차이를 계산
    optimizer.zero_grad()                   # 기울기 초기화
    loss.backward()                         # 역전파 계산 (자동 미분을 통해서 기울기의 방향을 알려준다.)
    optimizer.step()                        # 가중치 업데이트(가중치를 역전파의 계산 방향으로 이동)

    if (epoch + 1) % 30 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {round(loss.item(), 6)}')

Epoch 30/300, Loss: 320002336.0
Epoch 60/300, Loss: 293590880.0
Epoch 90/300, Loss: 202884704.0
Epoch 120/300, Loss: 87798624.0
Epoch 150/300, Loss: 55317616.0
Epoch 180/300, Loss: 43614552.0
Epoch 210/300, Loss: 42330628.0
Epoch 240/300, Loss: 39884184.0
Epoch 270/300, Loss: 41166408.0
Epoch 300/300, Loss: 40846336.0


In [220]:
# 모델 평가

model.eval()

with torch.no_grad():
    # no_grad(): 가중치 업데이트 잠시 중단
    pred = model(X_test_tensor)
    r2 = r2_score(y_test_tensor, pred)
    print(f'R2 Score: {round(r2, 4)}')
    print('예측값: ',pred[0], ' / 실제값: ',y_test_tensor[0])

R2 Score: 0.804
예측값:  tensor([6966.3677])  / 실제값:  tensor([9095.0684])


In [ ]:
# ML (XGBoost)을 이용하여 모델 학습 및 예측
# Scaler, Model Pipeline 구축

pipe = Pipeline(
    [
        ('std', StandardScaler()),
        ('xgb', XGBRegressor(random_state = 42))
    ]
)

params = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__max_depth': [3, 4, 5],
    'xgb__subsample': [0.8, 0.9, 1.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid= params,
    cv = cv,
    n_jobs = -1,
    scoring = 'r2'
)

grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'xgb__learning_rate': [0.01, 0.05, ...], 'xgb__max_depth': [3, 4, ...], 'xgb__n_estimators': [100, 200, ...], 'xgb__subsample': [0.8, 0.9, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- 

In [222]:
grid.best_params_

{'xgb__learning_rate': 0.05,
 'xgb__max_depth': 3,
 'xgb__n_estimators': 100,
 'xgb__subsample': 0.9}

In [224]:
pred = grid.predict(X_test)

r2 = r2_score(y_test, pred)
print(f'R2 Score: {round(r2, 4)}')
print('예측값: ',pred[0], '/ 실제값: ',y_test.values[0])

R2 Score: 0.8827
예측값:  10704.0625 / 실제값:  9095.06825
